# PROF-GRPO Final Implementation for AI4MATH Paper
================================================

**Paper**: "Does PROF Fix What It Claims to Fix?"

**P1+P2 Deliverables**:
1. Baseline GRPO training
2. PROF-GRPO training (Algorithm 1 from Ye et al.)
3. Checkpoints saved for P3 evaluation
4. Filtering statistics logged

**Hardware**: Kaggle 2xT4 (free tier)

**Run Order**: Execute cells 0-15 sequentially. Training cells can be run in separate sessions.

---
## Cell Index
- **Cells 0-2**: Setup & Dependencies
- **Cells 3-4**: Load Models (Policy + PRM)
- **Cells 5-6**: PRM Scoring Functions
- **Cells 7-8**: PROF Algorithm 1 (Official)
- **Cells 9-10**: Dataset & Reward Functions
- **Cell 11**: Smoke Test (MUST PASS)
- **Cell 12**: Baseline GRPO Training
- **Cell 13**: PROF-GRPO Training
- **Cell 14**: Save Checkpoints for P3
- **Cell 15**: Training Summary & Stats

In [ ]:
# ============================================================================
# CELL 0: INSTALL DEPENDENCIES (Tested & Working)
# ============================================================================
# Run time: ~3-5 minutes
# Matches working setup from probe-prof.ipynb

import sys
print(f"Python: {sys.version}")

# Step 1: Install core dependencies (NO -U flag to avoid breaking upgrades)
print("\n=== Step 1/4: Installing core packages ===")
!pip install datasets scipy accelerate peft bitsandbytes

# Step 2: Install specific transformers version (5.x works with Qwen PRM)
# The PRM model uses DynamicCache.from_legacy_cache which was removed in newer versions
print("\n=== Step 2/4: Installing transformers ===")
!pip install "transformers>=4.45.0,<4.50.0"

# Step 3: Install TRL with specific version (CRITICAL: must be <0.9.0)
print("\n=== Step 3/4: Installing TRL ===")
!pip install "trl<0.9.0"

# Step 4: Install Unsloth
print("\n=== Step 4/4: Installing Unsloth ===")
!pip install unsloth

print("\n" + "="*60)
print("ALL DEPENDENCIES INSTALLED")
print("="*60)

In [ ]:
# ============================================================================
# CELL 1: IMPORTS & ENVIRONMENT CHECK
# ============================================================================
# Verify all imports work after installation

import torch
import torch.nn.functional as F
import numpy as np
import random
import json
import re
import time
import os
from datetime import datetime
from collections import defaultdict

# These imports verify the installations worked
import transformers
import unsloth
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoTokenizer, AutoModel, AutoConfig
from datasets import load_dataset

print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Unsloth: {unsloth.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
    
# Create output directories
os.makedirs("./checkpoints/baseline_grpo", exist_ok=True)
os.makedirs("./checkpoints/prof_grpo", exist_ok=True)
os.makedirs("./logs", exist_ok=True)

print("\nEnvironment ready")

In [ ]:
# ============================================================================
# CELL 2: CONFIGURATION (EDIT FOR YOUR HARDWARE)
# ============================================================================

# === HARDWARE CONFIG ===
# For Kaggle 2xT4: Use these settings
# For L4/H100: Can increase batch sizes

CONFIG = {
    # Model settings
    "policy_model": "Qwen/Qwen2.5-Math-1.5B",
    "prm_model": "Qwen/Qwen2.5-Math-PRM-7B",
    
    # Training settings (Kaggle-optimized)
    "num_train_prompts": 1024,      # Per Ye et al.
    "num_generations": 8,            # n=8 rollouts per prompt
    "prof_keep": 4,                  # m=4 kept after PROF filtering
    "max_completion_length": 2048,   # Reduced for T4 memory
    "max_prompt_length": 512,
    "per_device_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 1e-6,
    "temperature": 1.0,
    "num_train_epochs": 1,
    
    # PROF hyperparameters (from official repo)
    "len_step_reward_coef": 10.0,    # Step length penalty
    "H_min": 2,                       # Min steps
    "H_max": 30,                      # Max steps (H_lambda)
    
    # Checkpointing
    "save_steps": 100,
    "logging_steps": 10,
}

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# ============================================================================
# CELL 3: LOAD POLICY MODEL (Qwen2.5-Math-1.5B)
# ============================================================================
# Run time: ~2-3 minutes
# VRAM: ~4-6 GB with LoRA

from unsloth import FastLanguageModel

print("Loading policy model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["policy_model"],
    max_seq_length=CONFIG["max_completion_length"] + CONFIG["max_prompt_length"],
    dtype=torch.float16,
    load_in_4bit=False,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=SEED,
)

# Record model revision for paper
POLICY_REVISION = "main"  # TODO: Pin to specific commit hash

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Policy model loaded")
print(f"  Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"  VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# ============================================================================
# CELL 4: LOAD PRM MODEL (Qwen2.5-Math-PRM-7B)
# ============================================================================
# Run time: ~3-5 minutes
# VRAM: ~8-10 GB in bf16

print("Loading PRM model...")

# Load and patch config (pad_token_id fix)
prm_config = AutoConfig.from_pretrained(
    CONFIG["prm_model"], 
    trust_remote_code=True
)
prm_config.pad_token_id = prm_config.eos_token_id

prm_tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["prm_model"], 
    trust_remote_code=True
)

prm_model = AutoModel.from_pretrained(
    CONFIG["prm_model"],
    config=prm_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
).eval()

# Freeze PRM (inference only)
for param in prm_model.parameters():
    param.requires_grad = False

# Record revision for paper
PRM_REVISION = "main"  # TODO: Pin to specific commit hash

print(f"PRM model loaded")
print(f"  VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Verify PRM works (use_cache=False to avoid DynamicCache issue)
print("\nVerifying PRM...")
test_input = prm_tokenizer("Test", return_tensors="pt").to(prm_model.device)
with torch.no_grad():
    test_out = prm_model(**test_input, use_cache=False)
print(f"  PRM output shape: {test_out.logits.shape}")
print(f"  PRM output has NaN: {torch.isnan(test_out.logits).any().item()}")
assert not torch.isnan(test_out.logits).any(), "PRM outputs NaN!"
print("  PRM verified OK")

In [ ]:
# ============================================================================
# CELL 5: PRM SCORING FUNCTIONS
# ============================================================================
# From official PROF-GRPO repo + Qwen PRM model card

def make_step_rewards(logits, token_masks):
    """
    Extract step-wise rewards from PRM logits.
    From Qwen2.5-Math-PRM model card.
    """
    probabilities = F.softmax(logits, dim=-1)
    probabilities = probabilities * token_masks.unsqueeze(-1)
    
    all_scores = []
    for i in range(probabilities.size(0)):
        sample = probabilities[i]
        positive_probs = sample[sample != 0].view(-1, 2)[:, 1]
        all_scores.append(positive_probs.cpu().tolist())
    return all_scores


def compute_prm_scores_batch(responses, prm_model, prm_tokenizer, batch_size=2):
    """
    Compute PRM step scores for responses.
    Memory-efficient batch processing.
    
    Returns:
        step_scores: List of step score lists
        mean_scores: List of mean PRM scores  
        num_steps: List of step counts
    """
    all_step_scores = []
    all_mean_scores = []
    all_num_steps = []
    
    for i in range(0, len(responses), batch_size):
        batch = responses[i:i + batch_size]
        
        for response in batch:
            # Split by \n\n (official step delimiter)
            steps = [s.strip() for s in response.split('\n\n') if s.strip()]
            
            if not steps:
                all_step_scores.append([])
                all_mean_scores.append(0.0)
                all_num_steps.append(0)
                continue
            
            # Format with <extra_0> markers (Qwen PRM format)
            messages = [
                {"role": "system", "content": "Please reason step by step."},
                {"role": "user", "content": "Solve this problem."},
                {"role": "assistant", "content": "<extra_0>".join(steps) + "<extra_0>"},
            ]
            
            conv = prm_tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            
            inputs = prm_tokenizer(conv, return_tensors="pt", padding=False)
            input_ids = inputs['input_ids'].to(prm_model.device)
            attn_mask = inputs['attention_mask'].to(prm_model.device)
            
            with torch.no_grad():
                # use_cache=False avoids DynamicCache compatibility issue
                outputs = prm_model(input_ids=input_ids, attention_mask=attn_mask, use_cache=False)
            
            logits = outputs.logits
            step_sep_id = prm_tokenizer.encode("<extra_0>")[0]
            token_masks = (input_ids == step_sep_id)
            
            step_rewards = make_step_rewards(logits, token_masks)
            scores = step_rewards[0] if step_rewards else []
            
            all_step_scores.append(scores)
            all_mean_scores.append(np.mean(scores) if scores else 0.0)
            all_num_steps.append(len(scores))
        
        torch.cuda.empty_cache()
    
    return all_step_scores, all_mean_scores, all_num_steps


# Test PRM scoring
print("Testing PRM scoring...")
test_resp = ["Step 1: Think.\n\nStep 2: Calculate.\n\nAnswer: 42"]
_, means, steps = compute_prm_scores_batch(test_resp, prm_model, prm_tokenizer)
print(f"  Test response: {steps[0]} steps, mean score: {means[0]:.4f}")
print("PRM scoring OK")

In [ ]:
# ============================================================================
# CELL 6: COMPUTE CONSISTENCY SCORES (Official PROF Algorithm)
# ============================================================================
# From: prof_grpo/prof_grpo_both/prof_ray_trainer.py

def compute_consistency_scores(prm_scores, outcome_rewards, num_steps):
    """
    Compute step-level consistency scores (official PROF).
    
    Formula: scores_ic = (score - batch_mean) * 2 * sign(outcome)
    
    Args:
        prm_scores: List of mean PRM scores per response
        outcome_rewards: List of {0, 1} (incorrect/correct)
        num_steps: List of step counts
    
    Returns:
        consistency_scores: List of consistency values
    """
    scores = np.array(prm_scores)
    outcomes = np.array(outcome_rewards)
    
    # Center scores around batch mean
    valid_mask = np.array(num_steps) > 0
    if valid_mask.sum() > 0:
        batch_mean = scores[valid_mask].mean()
    else:
        batch_mean = 0.0
    
    # (score - mean) * 2 * sign(outcome)
    # outcome: 0->-1, 1->+1
    outcome_sign = 2 * outcomes - 1
    consistency = (scores - batch_mean) * 2 * outcome_sign
    
    # Zero out invalid samples
    consistency[~valid_mask] = 0.0
    
    return consistency.tolist()


print("Consistency scoring function loaded")

In [ ]:
# ============================================================================
# CELL 7: PROF FILTERING ALGORITHM 1 (Official Implementation)
# ============================================================================
# From: prof_grpo/prof_grpo_both/prof_ray_trainer.py - filter_trajectories()

def prof_filter(
    responses,           # List of response texts
    outcome_rewards,     # List of {0, 1} (incorrect/correct)
    prm_scores,          # List of mean PRM scores
    num_steps,           # List of step counts
    uids,                # List of prompt UIDs (same uid = same prompt)
    n_remove=4,          # Number to remove per prompt group
    len_coef=10.0,       # Step length penalty coefficient
    H_min=2,             # Min acceptable steps
    H_max=30             # Max acceptable steps
):
    """
    PROF Algorithm 1: Balanced trajectory filtering.
    
    For each prompt group:
    1. Compute consistency with step length penalty
    2. Separate into correct (G+) and incorrect (G-)
    3. Remove lowest-scoring from G+, random from G-
    4. Balance removal to maintain outcome variance
    
    Returns:
        kept_indices: Indices of responses to keep
        stats: Filtering statistics dict
    """
    # Compute consistency scores
    consistency = compute_consistency_scores(prm_scores, outcome_rewards, num_steps)
    
    # Group by prompt UID
    uid_to_items = defaultdict(list)
    for idx, (uid, cons, n_step, outcome) in enumerate(zip(
        uids, consistency, num_steps, outcome_rewards
    )):
        # Apply step length penalty
        penalty = len_coef * ((n_step < H_min) or (n_step > H_max))
        critic_score = cons - penalty
        uid_to_items[uid].append((idx, critic_score, outcome))
    
    kept_indices = []
    stats = {"pos_removed": 0, "neg_removed": 0, "total_groups": 0}
    
    for uid, items in uid_to_items.items():
        stats["total_groups"] += 1
        n_total = len(items)
        n_rem = min(n_remove, n_total - 1)  # Keep at least 1
        
        if n_rem <= 0:
            kept_indices.extend([x[0] for x in items])
            continue
        
        # Separate G+ (correct) and G- (incorrect)
        pos = [x for x in items if x[2] == 1]
        neg = [x for x in items if x[2] == 0]
        
        # Calculate removal distribution (Eq. 2 from paper)
        delta = len(pos) - len(neg)
        n_pos_remove = min(n_rem, max(0, round((delta + n_rem) / 2)))
        n_neg_remove = n_rem - n_pos_remove
        
        # Remove lowest-scoring from G+ (by critic_score)
        if n_pos_remove > 0 and len(pos) > 0:
            pos_sorted = sorted(pos, key=lambda x: x[1])  # Ascending
            n_pos_remove = min(n_pos_remove, len(pos))
            remove_set = {x[0] for x in pos_sorted[:n_pos_remove]}
            pos = [x for x in pos if x[0] not in remove_set]
            stats["pos_removed"] += n_pos_remove
        
        # Remove random from G-
        if n_neg_remove > 0 and len(neg) > 0:
            n_neg_remove = min(n_neg_remove, len(neg))
            neg_remove = random.sample(neg, k=n_neg_remove)
            remove_set = {x[0] for x in neg_remove}
            neg = [x for x in neg if x[0] not in remove_set]
            stats["neg_removed"] += n_neg_remove
        
        kept_indices.extend([x[0] for x in pos])
        kept_indices.extend([x[0] for x in neg])
    
    stats["total_before"] = len(responses)
    stats["total_after"] = len(kept_indices)
    stats["filter_rate"] = 1 - len(kept_indices) / len(responses) if responses else 0
    
    return kept_indices, stats


# Test filtering
print("Testing PROF filtering...")
test_n = 8
test_kept, test_stats = prof_filter(
    responses=["r"] * test_n,
    outcome_rewards=[1, 1, 1, 1, 0, 0, 0, 0],
    prm_scores=[0.8, 0.6, 0.7, 0.5, 0.3, 0.4, 0.2, 0.1],
    num_steps=[5, 3, 4, 2, 3, 4, 5, 6],
    uids=[0] * test_n,
    n_remove=4
)
print(f"  Kept {len(test_kept)}/8 (expected: 4)")
print(f"  Stats: {test_stats}")
assert len(test_kept) == 4, "PROF filter should keep 4!"
print("PROF filtering OK")

In [ ]:
# ============================================================================
# CELL 8: PROF REWARD WRAPPER (Integrates with TRL GRPOTrainer)
# ============================================================================

class PROFRewardWrapper:
    """
    Wraps PROF filtering for use with TRL's GRPOTrainer.
    
    Strategy: Since GRPOTrainer doesn't support filtering,
    we give filtered responses reward=0 (neutral gradient).
    - Kept responses: true reward (+1 or -1)
    - Filtered responses: 0
    """
    
    def __init__(self, prm_model, prm_tokenizer, prompt_to_gt, config):
        # Required for GRPOTrainer compatibility (must be instance attribute)
        self.__name__ = "prof_reward_fn"
        
        self.prm_model = prm_model
        self.prm_tokenizer = prm_tokenizer
        self.prompt_to_gt = prompt_to_gt
        self.config = config
        
        # Statistics tracking
        self.stats = {
            "total_processed": 0,
            "total_filtered": 0,
            "pos_removed": 0,
            "neg_removed": 0,
            "correct_count": 0,
            "incorrect_count": 0,
        }
    
    def __call__(self, prompts, completions, **kwargs):
        n = len(prompts)
        self.stats["total_processed"] += n
        
        # Step 1: Compute outcome rewards
        outcome_rewards = []
        for prompt, completion in zip(prompts, completions):
            gt = self.prompt_to_gt.get(prompt)
            if gt is None:
                outcome_rewards.append(0)
            else:
                reward = compute_outcome_reward(completion, gt)
                outcome_rewards.append(1 if reward > 0 else 0)
        
        self.stats["correct_count"] += sum(outcome_rewards)
        self.stats["incorrect_count"] += n - sum(outcome_rewards)
        
        # Step 2: Compute PRM scores
        _, prm_scores, num_steps = compute_prm_scores_batch(
            completions, self.prm_model, self.prm_tokenizer, batch_size=2
        )
        
        # Step 3: Assign UIDs (hash of prompt)
        uids = [hash(p) for p in prompts]
        
        # Step 4: Apply PROF filtering
        n_remove = self.config["num_generations"] - self.config["prof_keep"]
        kept_indices, filter_stats = prof_filter(
            responses=completions,
            outcome_rewards=outcome_rewards,
            prm_scores=prm_scores,
            num_steps=num_steps,
            uids=uids,
            n_remove=n_remove,
            len_coef=self.config["len_step_reward_coef"],
            H_min=self.config["H_min"],
            H_max=self.config["H_max"]
        )
        
        self.stats["total_filtered"] += n - len(kept_indices)
        self.stats["pos_removed"] += filter_stats["pos_removed"]
        self.stats["neg_removed"] += filter_stats["neg_removed"]
        
        # Step 5: Build final rewards
        kept_set = set(kept_indices)
        final_rewards = []
        for i, (prompt, completion) in enumerate(zip(prompts, completions)):
            if i in kept_set:
                gt = self.prompt_to_gt.get(prompt)
                reward = compute_outcome_reward(completion, gt) if gt else -1.0
                final_rewards.append(float(reward))
            else:
                final_rewards.append(0.0)  # Filtered = neutral
        
        return final_rewards
    
    def get_stats(self):
        s = self.stats.copy()
        if s["total_processed"] > 0:
            s["filter_rate"] = s["total_filtered"] / s["total_processed"]
            s["accuracy"] = s["correct_count"] / s["total_processed"]
        return s
    
    def reset_stats(self):
        for k in self.stats:
            self.stats[k] = 0


print("PROF reward wrapper loaded")

In [ ]:
# ============================================================================
# CELL 9: LOAD DATASET & ANSWER EXTRACTION
# ============================================================================

def extract_boxed(text):
    """Extract \\boxed{...} content with nested brace handling."""
    match = re.search(r'\\boxed\{', text)
    if not match:
        return None
    start = match.end()
    depth = 1
    i = start
    while i < len(text) and depth > 0:
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
        i += 1
    return text[start:i-1].strip() if depth == 0 else None


def normalize_answer(expr):
    """Normalize LaTeX for comparison."""
    if expr is None:
        return None
    expr = re.sub(r'\s+', '', expr)
    expr = expr.replace('\\cdot', '*').replace('\\times', '*')
    return expr.lower()


def extract_answer(text):
    """Extract answer from model output."""
    if text is None:
        return None
    # Priority: \boxed{} > #### > "answer is"
    boxed = extract_boxed(text)
    if boxed:
        return boxed
    gsm = re.search(r'####\s*(.+?)(?:\n|$)', text)
    if gsm:
        return gsm.group(1).strip()
    ans = re.search(r'[Tt]he (?:final )?answer is[:\s]*(.+?)(?:\.|$)', text)
    if ans:
        return ans.group(1).strip()
    return None


def compute_outcome_reward(response, ground_truth):
    """
    Binary outcome reward: +1 if correct, -1 if incorrect.
    Per Ye et al.: ro in {-1, +1}
    """
    if ground_truth is None:
        return -1.0
    predicted = extract_answer(response)
    if predicted is None:
        return -1.0
    
    pred_norm = normalize_answer(predicted)
    gt_norm = normalize_answer(ground_truth)
    
    if pred_norm == gt_norm:
        return 1.0
    
    # Numeric fallback
    try:
        p = float(re.sub(r'[^\d.\-]', '', predicted))
        g = float(re.sub(r'[^\d.\-]', '', ground_truth))
        if abs(p - g) < 1e-6:
            return 1.0
    except:
        pass
    
    return -1.0


# Load NuminaMath dataset
print("Loading NuminaMath dataset...")
train_dataset = load_dataset("AI-MO/NuminaMath-CoT", split="train")

# Sample subset
random.seed(SEED)
indices = random.sample(range(len(train_dataset)), CONFIG["num_train_prompts"])
train_dataset = train_dataset.select(indices)

# Format prompts and extract ground truths
def format_prompt(ex):
    problem = ex.get("problem") or ex.get("question", "")
    return f"Question: {problem}\nAnswer:"

def get_ground_truth(ex):
    if "solution" in ex and ex["solution"]:
        boxed = extract_boxed(str(ex["solution"]))
        if boxed:
            return boxed
    return ex.get("answer")

train_prompts = [format_prompt(ex) for ex in train_dataset]
ground_truths = [get_ground_truth(ex) for ex in train_dataset]
prompt_to_gt = dict(zip(train_prompts, ground_truths))

valid_gt = sum(1 for gt in ground_truths if gt is not None)
print(f"Loaded {len(train_prompts)} prompts ({valid_gt} with valid GT)")
print(f"Sample prompt: {train_prompts[0][:100]}...")
print(f"Sample GT: {ground_truths[0]}")

In [ ]:
# ============================================================================
# CELL 10: REWARD FUNCTIONS FOR TRAINING
# ============================================================================

# Baseline reward function (simple lambda - works with Unsloth)
def baseline_reward_fn(prompts, completions, **kwargs):
    """Simple binary outcome reward for baseline GRPO."""
    rewards = []
    for prompt, completion in zip(prompts, completions):
        gt = prompt_to_gt.get(prompt)
        reward = compute_outcome_reward(completion, gt) if gt else -1.0
        rewards.append(float(reward))
    return rewards


# PROF reward - create wrapper instance for stats tracking
_prof_wrapper = PROFRewardWrapper(
    prm_model=prm_model,
    prm_tokenizer=prm_tokenizer,
    prompt_to_gt=prompt_to_gt,
    config=CONFIG
)

# Wrap in a regular function (Unsloth requires __name__ attribute)
def prof_reward_fn(prompts, completions, **kwargs):
    """PROF-filtered reward function."""
    return _prof_wrapper(prompts, completions, **kwargs)

# Helper to access stats
def get_prof_stats():
    return _prof_wrapper.get_stats()

def reset_prof_stats():
    _prof_wrapper.reset_stats()


# Test both rewards
print("Testing reward functions...")
test_p = [train_prompts[0]] * 2
test_c = [f"\\boxed{{{ground_truths[0]}}}", "\\boxed{{wrong}}"]

baseline_r = baseline_reward_fn(test_p, test_c)
print(f"  Baseline rewards: {baseline_r} (expected: [1.0, -1.0])")

reset_prof_stats()
prof_r = prof_reward_fn(test_p, test_c)
print(f"  PROF rewards: {prof_r}")

print("Reward functions OK")

In [ ]:
# ============================================================================
# CELL 11: SMOKE TEST (MUST PASS BEFORE TRAINING)
# ============================================================================
# This validates the entire pipeline works before committing to long training

print("="*60)
print("SMOKE TEST - Pipeline Validation")
print("="*60)

# Minimal config for smoke test
smoke_config = GRPOConfig(
    output_dir="./checkpoints/smoke_test",
    num_train_epochs=1,
    max_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=CONFIG["learning_rate"],
    num_generations=4,  # Reduced for smoke test
    temperature=CONFIG["temperature"],
    max_completion_length=512,  # Reduced
    max_prompt_length=256,
    logging_steps=1,
    report_to="none",
)

# Test with baseline reward first
print("\n[1/2] Testing baseline GRPO...")
smoke_trainer = GRPOTrainer(
    model=model,
    args=smoke_config,
    processing_class=tokenizer,
    reward_funcs=[baseline_reward_fn],
    train_dataset=[{"prompt": p} for p in train_prompts[:5]],
)

torch.cuda.empty_cache()
initial_mem = torch.cuda.memory_allocated() / 1e9

try:
    smoke_trainer.train()
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"  Baseline GRPO: PASSED")
    print(f"  Memory: {initial_mem:.2f} -> {peak_mem:.2f} GB")
except Exception as e:
    print(f"  Baseline GRPO: FAILED - {e}")
    raise

# Test with PROF reward
print("\n[2/2] Testing PROF-GRPO...")
reset_prof_stats()

smoke_trainer_prof = GRPOTrainer(
    model=model,
    args=smoke_config,
    processing_class=tokenizer,
    reward_funcs=[prof_reward_fn],
    train_dataset=[{"prompt": p} for p in train_prompts[:5]],
)

torch.cuda.empty_cache()

try:
    smoke_trainer_prof.train()
    print(f"  PROF-GRPO: PASSED")
    print(f"  PROF stats: {get_prof_stats()}")
except Exception as e:
    print(f"  PROF-GRPO: FAILED - {e}")
    raise

print("\n" + "="*60)
print("SMOKE TEST PASSED - Ready for full training")
print("="*60)

In [ ]:
# ============================================================================
# CELL 12: BASELINE GRPO TRAINING
# ============================================================================
# Run time: ~6-8 hours on 2xT4 (Kaggle)
# Checkpoint saved to: ./checkpoints/baseline_grpo/

print("="*60)
print("BASELINE GRPO TRAINING")
print("="*60)
print(f"Start time: {datetime.now()}")
print(f"Prompts: {len(train_prompts)}")
print(f"Generations/prompt: {CONFIG['num_generations']}")
print("="*60)

baseline_config = GRPOConfig(
    output_dir="./checkpoints/baseline_grpo",
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    num_generations=CONFIG["num_generations"],
    temperature=CONFIG["temperature"],
    max_completion_length=CONFIG["max_completion_length"],
    max_prompt_length=CONFIG["max_prompt_length"],
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=3,
    fp16=True,
    report_to="none",
)

baseline_trainer = GRPOTrainer(
    model=model,
    args=baseline_config,
    processing_class=tokenizer,
    reward_funcs=[baseline_reward_fn],
    train_dataset=[{"prompt": p} for p in train_prompts],
)

# Training
start_time = time.time()
try:
    baseline_trainer.train()
    elapsed = time.time() - start_time
    
    # Save final checkpoint
    model.save_pretrained("./checkpoints/baseline_grpo/final")
    tokenizer.save_pretrained("./checkpoints/baseline_grpo/final")
    
    # Save metadata
    metadata = {
        "type": "baseline_grpo",
        "model": CONFIG["policy_model"],
        "model_revision": POLICY_REVISION,
        "training_time_hours": elapsed / 3600,
        "final_step": baseline_trainer.state.global_step,
        "config": CONFIG,
        "timestamp": datetime.now().isoformat(),
    }
    with open("./checkpoints/baseline_grpo/metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    
    print(f"\nBaseline GRPO completed in {elapsed/3600:.2f} hours")
    print(f"Checkpoint saved to ./checkpoints/baseline_grpo/final")
    
except KeyboardInterrupt:
    print("\nTraining interrupted - checkpoint saved at last save_step")
except Exception as e:
    print(f"\nTraining failed: {e}")
    raise

In [ ]:
# ============================================================================
# CELL 13: PROF-GRPO TRAINING
# ============================================================================
# Run time: ~8-10 hours on 2xT4 (Kaggle) - slower due to PRM scoring
# Checkpoint saved to: ./checkpoints/prof_grpo/

print("="*60)
print("PROF-GRPO TRAINING (Algorithm 1)")
print("="*60)
print(f"Start time: {datetime.now()}")
print(f"Prompts: {len(train_prompts)}")
print(f"Generations/prompt: {CONFIG['num_generations']} -> {CONFIG['prof_keep']} after filter")
print(f"Step penalty coef: {CONFIG['len_step_reward_coef']}")
print(f"Step range: [{CONFIG['H_min']}, {CONFIG['H_max']}]")
print("="*60)

# Reset PROF stats
reset_prof_stats()

prof_config = GRPOConfig(
    output_dir="./checkpoints/prof_grpo",
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    num_generations=CONFIG["num_generations"],
    temperature=CONFIG["temperature"],
    max_completion_length=CONFIG["max_completion_length"],
    max_prompt_length=CONFIG["max_prompt_length"],
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=3,
    fp16=True,
    report_to="none",
)

prof_trainer = GRPOTrainer(
    model=model,
    args=prof_config,
    processing_class=tokenizer,
    reward_funcs=[prof_reward_fn],
    train_dataset=[{"prompt": p} for p in train_prompts],
)

# Training
start_time = time.time()
try:
    prof_trainer.train()
    elapsed = time.time() - start_time
    
    # Save final checkpoint
    model.save_pretrained("./checkpoints/prof_grpo/final")
    tokenizer.save_pretrained("./checkpoints/prof_grpo/final")
    
    # Save metadata with PROF stats
    prof_stats = get_prof_stats()
    metadata = {
        "type": "prof_grpo",
        "model": CONFIG["policy_model"],
        "model_revision": POLICY_REVISION,
        "prm_model": CONFIG["prm_model"],
        "prm_revision": PRM_REVISION,
        "training_time_hours": elapsed / 3600,
        "final_step": prof_trainer.state.global_step,
        "prof_stats": prof_stats,
        "config": CONFIG,
        "timestamp": datetime.now().isoformat(),
    }
    with open("./checkpoints/prof_grpo/metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    
    # Save detailed filtering log
    with open("./logs/prof_filtering_stats.json", "w") as f:
        json.dump(prof_stats, f, indent=2)
    
    print(f"\nPROF-GRPO completed in {elapsed/3600:.2f} hours")
    print(f"Checkpoint saved to ./checkpoints/prof_grpo/final")
    print(f"\nPROF Filtering Statistics:")
    for k, v in prof_stats.items():
        print(f"  {k}: {v}")
    
except KeyboardInterrupt:
    print("\nTraining interrupted - saving current stats")
    with open("./logs/prof_filtering_stats_partial.json", "w") as f:
        json.dump(get_prof_stats(), f, indent=2)
except Exception as e:
    print(f"\nTraining failed: {e}")
    raise

In [ ]:
# ============================================================================
# CELL 14: SAVE CHECKPOINTS FOR P3 EVALUATION
# ============================================================================

print("="*60)
print("Preparing checkpoints for P3 evaluation")
print("="*60)

import shutil

# Create P3 handoff package
os.makedirs("./p3_handoff", exist_ok=True)

# Copy checkpoints
if os.path.exists("./checkpoints/baseline_grpo/final"):
    shutil.copytree("./checkpoints/baseline_grpo/final", "./p3_handoff/baseline_grpo", dirs_exist_ok=True)
    print("Copied baseline_grpo checkpoint")

if os.path.exists("./checkpoints/prof_grpo/final"):
    shutil.copytree("./checkpoints/prof_grpo/final", "./p3_handoff/prof_grpo", dirs_exist_ok=True)
    print("Copied prof_grpo checkpoint")

# Copy metadata and logs
for f in ["./checkpoints/baseline_grpo/metadata.json", 
          "./checkpoints/prof_grpo/metadata.json",
          "./logs/prof_filtering_stats.json"]:
    if os.path.exists(f):
        shutil.copy(f, "./p3_handoff/")

# Create P3 instructions
p3_readme = """
# P3 Evaluation Handoff

## Checkpoints
- `baseline_grpo/`: Baseline GRPO model (no PROF filtering)
- `prof_grpo/`: PROF-GRPO model (with Algorithm 1 filtering)

## Evaluation Tasks for P3
1. **Average@16 accuracy** on: MATH-500, OlympiadBench, Minerva Math, AMC23, AIME
2. **Pass@k** (k=1,4,8,32) on above benchmarks
3. **RAC scoring** with Gemma-3-4B-IT and Llama-3.1-8B-Instruct judges
4. **Memorization probe** (n-gram overlap check)

## Model Loading
```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Math-1.5B")
model = PeftModel.from_pretrained(base_model, "./p3_handoff/prof_grpo")
tokenizer = AutoTokenizer.from_pretrained("./p3_handoff/prof_grpo")
```

## Metadata
See `metadata.json` files for training config and PROF filtering stats.
"""

with open("./p3_handoff/README.md", "w") as f:
    f.write(p3_readme)

print("\nP3 handoff package ready at ./p3_handoff/")
print("Contents:")
for item in os.listdir("./p3_handoff"):
    print(f"  - {item}")

In [ ]:
# ============================================================================
# CELL 15: TRAINING SUMMARY
# ============================================================================

print("="*60)
print("P1+P2 DELIVERABLES COMPLETE")
print("="*60)

# Load and display metadata
summary = {
    "baseline_grpo": None,
    "prof_grpo": None,
}

for name in summary.keys():
    meta_path = f"./checkpoints/{name}/metadata.json"
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            summary[name] = json.load(f)

print("\n[1] Baseline GRPO:")
if summary["baseline_grpo"]:
    m = summary["baseline_grpo"]
    print(f"    Training time: {m.get('training_time_hours', 'N/A'):.2f} hours")
    print(f"    Final step: {m.get('final_step', 'N/A')}")
    print(f"    Checkpoint: ./checkpoints/baseline_grpo/final")
else:
    print("    Not completed")

print("\n[2] PROF-GRPO:")
if summary["prof_grpo"]:
    m = summary["prof_grpo"]
    print(f"    Training time: {m.get('training_time_hours', 'N/A'):.2f} hours")
    print(f"    Final step: {m.get('final_step', 'N/A')}")
    print(f"    Checkpoint: ./checkpoints/prof_grpo/final")
    if "prof_stats" in m:
        ps = m["prof_stats"]
        print(f"    PROF Filter rate: {ps.get('filter_rate', 0)*100:.1f}%")
        print(f"    Correct removed: {ps.get('pos_removed', 'N/A')}")
        print(f"    Incorrect removed: {ps.get('neg_removed', 'N/A')}")
else:
    print("    Not completed")

print("\n[3] P3 Handoff:")
print(f"    Package: ./p3_handoff/")
print(f"    Ready for evaluation")

print("\n" + "="*60)
print("Next: P3 runs evaluation (Pass@k, RAC, benchmarks)")
print("="*60)